---
short_title: Excel einlesen
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Excel-Dateien einlesen

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

Bibliothekssysteme liefern ihre Exporte häufig als {term}`Excel`-Arbeitsmappe
(Workbook) im Format `.xlsx` – im Arbeitsalltag das verbreitetste
Austauschformat neben CSV.

Für die Auswertung nutzen wir die {term}`Bibliothek` **{term}`Pandas`**. Zentral ist das
{term}`DataFrame`: eine zweidimensionale Tabelle mit benannten Spalten (Columns) und
einem Zeilenindex (Index), die einer Excel-Tabelle sehr ähnlich ist und
statistische Auswertungen in wenigen Zeilen erlaubt.

```{seealso} Vorwissen
:icon: false

Die Grundlagen zum Umgang mit Dateien und zum Format CSV haben Sie in
[Projekt CSV I](../040-Projekt_CSV_I/000-Einleitung.md) und
[Projekt CSV II](../070-Projekt_CSV_II/000-Einleitung.md) kennengelernt.
```

In [1]:
import pandas as pd

```{note}
**Hinweis zur Browser-Version:** Der Python-Kernel von JupyterLite (über den
Start-Knopf der Seite) läuft im Browser und hat ein anderes Environment als
Ihre lokale Installation: Die Beispieldatei
`assets/data/bibliothek_unsauber.xlsx` liegt dort nicht im Projekt, und die
Excel-Engine `openpyxl` fehlt. Die folgende Zelle lädt beides **nur im
Browser** von der Kurs-Website nach (erkannt an
`sys.platform == "emscripten"`). Lokal in JupyterLab tut sie nichts.
```

In [ ]:
import sys

# Pyodide (JupyterLite) meldet sich mit der Plattform "emscripten";
# lokal (JupyterLab, Build) ist sie z. B. "darwin" oder "linux".
if sys.platform == "emscripten":
    from pathlib import Path

    from js import location

    # 1) Excel-Engine openpyxl fehlt in JupyterLite. Die Wheels liegen auf
    #    der Kurs-Website (Ordner assets/pypi) und werden direkt installiert.
    import micropip

    _wheels = (
        "et_xmlfile-2.0.0-py3-none-any.whl",
        "openpyxl-3.1.5-py2.py3-none-any.whl",
    )
    _fehler = None
    for _basis in (f"{location.origin}/pypi", "pypi"):
        try:
            await micropip.install(
                [f"{_basis}/{_w}" for _w in _wheels], keep_going=True
            )
            _fehler = None
            break
        except Exception as _e:  # noqa: BLE001
            _fehler = _e
    if _fehler is not None:
        raise RuntimeError(f"openpyxl konnte nicht installiert werden: {_fehler}")

    # 2) Beispieldatei von der Kurs-Website ins virtuelle Dateisystem laden,
    #    sodass relative Pfade wie "../assets/data/..." funktionieren
    def stelle_daten_bereit(pfad):
        """Lädt eine Beispieldatei von der Kurs-Website ins virtuelle Dateisystem."""
        ziel = Path(pfad)
        if ziel.exists():
            return ziel

        from js import XMLHttpRequest

        for url in (f"{location.origin}/{ziel.name}", ziel.name):
            anfrage = XMLHttpRequest.new()
            # synchron möglich, weil der Kernel in einem Web Worker läuft
            anfrage.open("GET", url, False)
            try:
                anfrage.responseType = "arraybuffer"
                anfrage.send()
            except Exception:
                continue  # nächste Kandidaten-URL versuchen
            if anfrage.status == 200:
                inhalt = anfrage.response.to_py()
                if not isinstance(inhalt, str):  # ArrayBuffer -> Bytes
                    ziel.parent.mkdir(parents=True, exist_ok=True)
                    ziel.write_bytes(bytes(inhalt))
                    return ziel
        raise FileNotFoundError(
            f"{pfad} weder lokal noch auf der Kurs-Website gefunden."
        )

    stelle_daten_bereit("../assets/data/bibliothek_unsauber.xlsx")

    # 3) Ausgabeverzeichnis für die Abbildungen sicherstellen
    Path("../assets/090").mkdir(parents=True, exist_ok=True)

## Die Beispieldatei

Wir arbeiten mit einer bewusst "unsauberen" Datei,
`assets/data/bibliothek_unsauber.xlsx`. Sie enthält bibliothekarische Daten
(Titel, Autor\*in, Erscheinungsjahr, Ausleihzahlen, Standort, Medium,
Zugangsdatum) mit typischen Problemen:

* fehlende Werte,
* doppelte Zeilen,
* Zahlen, die als Text gespeichert sind,
* uneinheitliche Schreibweisen bei Kategorien.

Die Datei wird reproduzierbar mit dem Skript
[`generate_excel_data.py`](generate_excel_data.py) erzeugt (fester Zufallsseed).

In [2]:
datei = "../assets/data/bibliothek_unsauber.xlsx"
pd.ExcelFile(datei).sheet_names

['Ausleihen', 'Standorte']

## Ein Tabellenblatt auswählen

Eine Excel-Datei kann mehrere Tabellenblätter (Sheets) enthalten. Ohne
`sheet_name` liest pandas das **erste** Blatt. Mit `sheet_name` wählen Sie ein
Blatt gezielt aus.

In [3]:
df = pd.read_excel(datei, sheet_name="Ausleihen")
df.head()

,Titel,Autor*in,Erscheinungsjahr,Ausleihzahlen,Standort,Medium,Zugangsdatum
0,Einführung in die Bibliothekswissenschaft,NaN,2005,382,zentralbibliothek,buch,22.02.2005
1,Metadaten in der Praxis,"Behrens, Carl",2007,8,Magazin,dvd,2007-03-23
2,Informationskompetenz vermitteln,NaN,2014,3401,zentralbibliothek,DVD,2014-11-20
3,Grundlagen der Katalogisierung,"Dittrich, Erik",unbekannt,186,West,zeitschrift,22.04.2013
4,Digitalisierung im Archiv,"Engel, Friederike",1993,21,MAG,eBook,1993-06-13


## Erste Einblicke

Verschaffen Sie sich immer zuerst einen Überblick: `shape` liefert
(Zeilen, Spalten), `info()` zeigt Spalten, Datentypen und fehlende Werte, und
`dtypes` gibt nur die Datentypen aus.

In [4]:
df.shape

(65, 7)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Titel             64 non-null     str   
 1   Autor*in          58 non-null     str   
 2   Erscheinungsjahr  56 non-null     object
 3   Ausleihzahlen     60 non-null     object
 4   Standort          65 non-null     str   
 5   Medium            65 non-null     str   
 6   Zugangsdatum      59 non-null     str   
dtypes: object(2), str(5)
memory usage: 3.7+ KB


In [6]:
df.dtypes

Titel                  str
Autor*in               str
Erscheinungsjahr    object
Ausleihzahlen       object
Standort               str
Medium                 str
Zugangsdatum           str
dtype: object

## Typ-Probleme erkennen

`object` bedeutet, dass pandas in einer Spalte Text vermutet. Für
`Erscheinungsjahr` ist das ein Warnsignal: Eigentlich erwarten wir Zahlen.

In [7]:
df["Erscheinungsjahr"].unique()

array(['2005 ', '2007 ', 2014, 'unbekannt', 1993, '2024 ', 2006, nan,
       2000, 2020, 2008, 2005, 2009, 2024, 2011, 2015, 1998, 2007, 2016,
       1991, 1994, 1990, 2013, 2021, '1996 ', '2008 ', 2023, '2021 ',
       2019], dtype=object)

In [8]:
df["Ausleihzahlen"].apply(type).value_counts()

Ausleihzahlen
<class 'int'>      49
<class 'str'>      11
<class 'float'>     5
Name: count, dtype: int64

In `Ausleihzahlen` stehen also gemischte Werte: ganze Zahlen **und** Text. Bei
deutschen Zahlen mit Tausenderpunkt (z. B. `8.798`) droht eine stille
Fehlinterpretation als Dezimalzahl `8.798`. Das schauen wir uns genauer an.

In [9]:
df.loc[
    df["Ausleihzahlen"].astype("string").str.contains(r"\.", na=False),
    "Ausleihzahlen",
]

33    8.798
52    7.100
Name: Ausleihzahlen, dtype: object

## Ein zweites Tabellenblatt

Die Standort-Nachschlagetabelle ist sauber und eignet sich später zum
Verknüpfen (Join).

In [10]:
standorte = pd.read_excel(datei, sheet_name="Standorte")
standorte

,Standort,Adresse,Öffnungsstunden_pro_Woche
0,Zentralbibliothek,Universitätsstraße 1,72
1,Zweigstelle Nord,Nordallee 12,40
2,Zweigstelle Süd,Südring 7,40
3,Campus West,Campusweg 3,35
4,Magazin,Depotstraße 99,0


## Übung

````{exercise}
:label: excel-einlesen-uebung

1. Lesen Sie nur die Spalten `Titel`, `Medium` und `Ausleihzahlen` ein
   (Tipp: Parameter `usecols`).
2. Ermitteln Sie mit `df["Standort"].nunique()`, wie viele verschiedene
   Standort-Schreibweisen es gibt.
3. Vergleichen Sie das Ergebnis mit der Nachschlagetabelle `standorte`.

````